# infra-defect-detection — Phase 1 on Kaggle,v3(处理FigShare的zip-of-zips结构)

**v2实际跑起来之后发现的新情况**:FigShare这个"合并压缩包"其实不是六国VOC文件的扁平合集,而是**zip里嵌套了7个zip**——顶层只有7个entry:`RDD2022/Japan.zip`、`RDD2022/India.zip`、`RDD2022/Czech.zip`、`RDD2022/Norway.zip`、`RDD2022/United_States.zip`、`RDD2022/China_MotorBike.zip`、`RDD2022/China_Drone.zip`,每一个本身又是一个完整的国家zip。v2按"目录名精确匹配国家别名"去找文件,但实际匹配到的是文件名(`Japan.zip`)而不是目录名(`Japan`),所以7个国家全部匹配失败,转换出来的`manifest.csv`是空的——这也是你跑`eda_report.py`时报`KeyError: 'country'`的直接原因(空DataFrame没有country这一列)。

**v3的修复**:改成按**文件名去掉`.zip`后缀**匹配国家(`Path('RDD2022/Japan.zip').stem == 'Japan'`),然后每个国家单独处理:把这个国家的嵌套zip从外层zip里复制出来→解压这个小zip→转换成YOLO格式→删掉解压出来的原始数据和这个临时的嵌套zip→再处理下一个国家。峰值占用还是控制在"外层zip(12.35GB全程不变)+当前一个国家的嵌套zip+原始数据+转换数据"这个量级,没有改变v2解决磁盘配额问题的思路,只是把"按国家分批"这一步改成了先处理"zip里的zip"这一层。

我已经用一个模拟了真实zip-of-zips结构(7个国家、每个都是嵌套zip、内部train/images+train/annotations/xmls)的小fixture,在沙盒里把这一版**从下载后的解压步骤到EDA完整跑通了一遍**,确认manifest.csv不再是空的、eda_report.py能正常出报告和图表,再交给你在Kaggle上跑真实数据。

**跑之前**:右侧Settings面板把 `Internet` 打开;Accelerator保持 `None`。

**如果你是接着上次报错的session继续**:12.35GB的zip应该还在(下载脚本已经跑过、没被删),不用重新下载——直接从下面的"第4步"(`extract_convert_per_country.py`)开始跑就行,不用重跑第3步的下载。如果不确定,先跑第0步看一眼`data/zips/`里那个zip还在不在。


## 0. 检查磁盘空间 + 确认zip是否还在

In [ ]:
!df -h /
!nproc
!free -h
!echo '--- data/zips ---'
!ls -la data/zips/ 2>&1


## 1. 写入四个脚本(前三个和本地Mac仓库一致;`extract_convert_per_country.py`是v3版本,处理zip-of-zips结构)

In [ ]:
!mkdir -p scripts


In [ ]:
%%writefile scripts/download_rdd2022.py
"""Download and extract the RDD2022 road damage dataset (roadmap: infra-defect-detection, phase 1).

RDD2022 covers six countries (Japan, India, Czech Republic, Norway, United States, China) with
47,420 road images and 55,000+ annotated damage instances across four classes (D00 longitudinal
crack, D10 transverse crack, D20 alligator crack, D40 pothole). Images are CC BY-SA 4.0 per the
sekilab/RoadDamageDetector GitHub README (attribute sekilab/RoadDamageDetector + the RDD2022 paper,
arxiv.org/abs/2209.08538, in any README/Model Card that uses this data) - note the FigShare listing
below shows "CC BY 4.0" for the same dataset; this discrepancy between the two official sources is
unresolved, so treat CC BY-SA 4.0 (the more restrictive of the two, and the one stated by the
dataset's own authors on their own repo) as the operative license until/unless clarified.

SOURCE CHANGE (2026-09-16): this originally downloaded seven per-country zips from Sekilab's own S3
bucket (bigdatacup.s3.ap-northeast-1.amazonaws.com/.../Country_Specific_Data_CRDDC2022/...). That
bucket now returns HTTP 403 Forbidden on every file, confirmed independently from two unrelated
networks - the bucket's access policy appears to have changed, not a transient fluke. This version
instead downloads FigShare's official combined mirror of the same dataset (one zip, all six
countries, published by the RDD2022/CRDDC2022 organizers themselves), verified by MD5 checksum
against FigShare's own published hash so a partial/corrupted 12GB+ download is caught rather than
silently producing bad data. If FigShare's link ever breaks too, check
https://github.com/sekilab/RoadDamageDetector for current mirror links before assuming this script
is broken.

Because this is one combined zip (not one zip per country), there's no way to download only a
subset of countries - the whole ~12.35GB has to come down regardless. --countries now only controls
which countries get linked into data/raw/ for the conversion step afterward (convert_voc_to_yolo.py
reads data/raw/<country>/), not what gets downloaded.

The zip's exact internal folder layout was not independently verified before writing this script
(Sekilab's own "Directory_Structure_CRDDC_RDD2022.txt" reference file was unreachable from this
environment - see reorganize_countries() below for how this script copes with that uncertainty at
extraction time instead of assuming a fixed layout).

NOTE (2026-09-17, Kaggle handoff): on the original Mac/home-network run, aria2c's multi-connection
mode was confirmed to get an immediate HTTP 403 from FigShare's CDN regardless of connection count
(1, 4, or 16) - the CDN appears to reject Range/segmented requests outright, not just high
concurrency. So on Kaggle this will (correctly) fall back to the plain single-connection urllib
path every time; that's expected, not a bug - the win here is Kaggle's raw single-connection
bandwidth to this host, not multi-connection parallelism.

Usage:
    python scripts/download_rdd2022.py                        # download + extract + link all found countries
    python scripts/download_rdd2022.py --countries Japan Czech # only link these two after extraction
    python scripts/download_rdd2022.py --skip-extract          # download (+ verify) only
    python scripts/download_rdd2022.py --connections 32        # more aria2c connections (default 16)
"""
import argparse
import hashlib
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

FIGSHARE_URL = "https://ndownloader.figshare.com/files/38030910"
FIGSHARE_ZIP_NAME = "RDD2022_released_through_CRDDC2022.zip"
FIGSHARE_MD5 = "b62bd51d2ffcfaa76c60f234f0cc2bb3"

# Logical country name -> name(s) we'll look for (case-insensitively) among directories inside the
# extracted zip, since the exact internal layout wasn't independently verified (see module docstring).
COUNTRY_ALIASES = {
    "Japan": ["Japan"],
    "India": ["India"],
    "Czech": ["Czech"],
    "Norway": ["Norway"],
    "United_States": ["United_States", "United States", "US", "USA"],
    "China_MotorBike": ["China_MotorBike", "China-MotorBike", "China_Motorbike"],
    "China_Drone": ["China_Drone", "China-Drone"],
}

DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"


def _format_bytes(n):
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}TB"


def download_with_aria2(url, dest_path, connections=16, retries=3):
    """Segmented, multi-connection download via the aria2c CLI - typically several times faster than
    a single urllib stream on hosts like FigShare's S3-backed CDN, where per-connection throughput
    is often capped well below the link's actual bandwidth. aria2c handles its own retries/resume
    (via its .aria2 control file next to the output), so this just shells out and lets it manage
    that; --continue=true means re-running after an interrupted aria2c download resumes rather than
    restarting from zero.
    """
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "aria2c",
        "-x", str(connections),          # max connections per server
        "-s", str(connections),          # split file into this many pieces
        "-k", "1M",                      # min split size
        "--continue=true",
        "--max-tries", str(retries),
        "--retry-wait=5",
        "--file-allocation=none",
        "--summary-interval=5",
        # FigShare's CDN (via ndownloader.figshare.com -> a presigned S3 URL) returns 403 to aria2c's
        # default "aria2/x.y.z" User-Agent - matching the header the plain-urllib path already used
        # successfully fixes it. --auto-file-renaming=false avoids aria2 silently writing to a
        # "-1" suffixed file if dest_path.name already exists from an earlier failed attempt.
        "--user-agent=Mozilla/5.0",
        "--auto-file-renaming=false",
        "-d", str(dest_path.parent),
        "-o", dest_path.name,
        url,
    ]
    print(f"  using aria2c with {connections} parallel connections (much faster than a single stream)")
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f"aria2c exited with code {result.returncode} - see its output above for details")


def download_one(url, dest_path, retries=3, connections=16):
    """Download url to dest_path. Skips entirely if dest_path already exists and is non-empty - this
    does NOT check the existing file's checksum, so a corrupted prior download won't be caught here;
    verify_md5() below is what actually guarantees integrity, run separately after this.

    Uses aria2c (multi-connection, much faster) when it's installed on PATH; otherwise falls back to
    a plain single-connection urllib download and prints a one-time hint about installing aria2c for
    a large file like this one. `brew install aria2` on macOS. If aria2c is present but fails (a
    misbehaving CDN, a network that blocks it, etc.), falls back to the urllib path automatically
    rather than giving up outright - not verified against every possible aria2c/network combination,
    so this fallback matters.
    """
    if dest_path.exists() and dest_path.stat().st_size > 0:
        print(f"  already have {dest_path.name} ({_format_bytes(dest_path.stat().st_size)}), skipping download")
        return

    dest_path.parent.mkdir(parents=True, exist_ok=True)

    if shutil.which("aria2c"):
        try:
            download_with_aria2(url, dest_path, connections=connections, retries=retries)
            return
        except RuntimeError as exc:
            print(f"  aria2c failed ({exc}); falling back to a plain single-connection download instead.")
            # clean up whatever partial/zero-byte file aria2c may have left behind before falling back
            if dest_path.exists() and dest_path.stat().st_size == 0:
                dest_path.unlink()
    else:
        print("  NOTE: aria2c not found on PATH - using a single-connection download, which will be "
              "noticeably slower for a file this size. `brew install aria2` (macOS) and re-run for a "
              "multi-connection download instead.")

    _download_urllib(url, dest_path, retries=retries)


def _download_urllib(url, dest_path, retries=3):
    """Plain single-connection streaming download - the fallback used when aria2c isn't available or
    didn't work."""
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")

    for attempt in range(1, retries + 1):
        try:
            print(f"  downloading {url} -> {dest_path} (attempt {attempt}/{retries})")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as resp, open(tmp_path, "wb") as out:
                total = int(resp.headers.get("Content-Length", 0))
                downloaded = 0
                chunk_size = 1024 * 1024
                last_print = time.time()
                while True:
                    chunk = resp.read(chunk_size)
                    if not chunk:
                        break
                    out.write(chunk)
                    downloaded += len(chunk)
                    if time.time() - last_print > 2:
                        pct = f"{downloaded / total:.0%}" if total else "?"
                        print(f"    {_format_bytes(downloaded)}" + (f" / {_format_bytes(total)} ({pct})" if total else ""),
                              end="\r", file=sys.stderr)
                        last_print = time.time()
            tmp_path.rename(dest_path)
            print(f"\n  done: {dest_path.name} ({_format_bytes(dest_path.stat().st_size)})")
            return
        except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, ConnectionError) as exc:
            print(f"\n  attempt {attempt} failed: {exc}")
            if tmp_path.exists():
                tmp_path.unlink()
            if attempt == retries:
                raise
            time.sleep(3 * attempt)


def verify_md5(path, expected_md5):
    """Stream-hash path and compare against expected_md5. Caches a good result next to the file
    (a .md5ok marker) so re-running this script doesn't re-hash a ~12GB file every time - if you
    ever suspect the downloaded file got corrupted after the fact, delete the .md5ok marker (or the
    zip itself) to force a real re-check.
    """
    marker = path.with_suffix(path.suffix + ".md5ok")
    if marker.exists():
        print(f"  {path.name}: MD5 previously verified (delete {marker.name} to re-check)")
        return True

    print(f"  verifying MD5 of {path.name} ({_format_bytes(path.stat().st_size)}, this can take a minute)...")
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(8 * 1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    actual = h.hexdigest()
    if actual != expected_md5:
        print(f"  MD5 MISMATCH: expected {expected_md5}, got {actual}")
        print(f"  {path} is corrupt or incomplete - delete it and re-run this script to re-download.")
        return False
    marker.write_text("ok\n")
    print(f"  MD5 verified: {actual}")
    return True


def extract_one(zip_path, extract_root):
    """Extract zip_path into extract_root, skipping if already extracted (marker-based, not by
    checking for a specific internal folder name - see reorganize_countries() for why)."""
    marker = extract_root / ".extracted"
    if marker.exists():
        print(f"  already extracted into {extract_root}, skipping")
        return
    extract_root.mkdir(parents=True, exist_ok=True)
    print(f"  extracting {zip_path.name} -> {extract_root} (large archive, this can take several minutes)")
    with zipfile.ZipFile(zip_path) as zf:
        bad = zf.testzip()
        if bad is not None:
            raise RuntimeError(f"{zip_path} is corrupt (bad member: {bad}) - delete it and re-run to re-download")
        zf.extractall(extract_root)
    marker.write_text("ok\n")


def reorganize_countries(extract_root, raw_dir, wanted_countries):
    """Find each wanted country's directory somewhere inside the extracted tree and link it to
    raw_dir/<country>, so convert_voc_to_yolo.py (which expects data/raw/<country>/ folders) keeps
    working unchanged regardless of whatever top-level wrapper folder(s) FigShare's zip actually
    uses internally.

    Uses a symlink rather than copying, since the extracted tree is already ~12GB+ and duplicating
    it serves no purpose. For each country, searches for directories whose name matches one of its
    known aliases (case-insensitive) and picks the SHALLOWEST match; if more than one directory at
    that same shallowest depth matches (e.g. the zip ships both a train/ and test/ split each with
    their own per-country subfolder), this prints every candidate found and picks the first
    alphabetically - flagged clearly so you can sanity-check it, rather than silently guessing.
    This is exactly the "raw data doesn't match documentation" messiness this project is meant to
    surface, so warn rather than hide it.
    """
    raw_dir.mkdir(parents=True, exist_ok=True)
    found_any = False

    for country in wanted_countries:
        link_path = raw_dir / country
        if link_path.exists() or link_path.is_symlink():
            print(f"  {country}: {link_path} already exists, leaving as-is")
            found_any = True
            continue

        aliases_lower = {a.lower() for a in COUNTRY_ALIASES[country]}
        candidates = [
            p for p in extract_root.rglob("*")
            if p.is_dir() and p.name.lower() in aliases_lower
        ]
        if not candidates:
            print(f"  WARNING: no directory matching {COUNTRY_ALIASES[country]} found under {extract_root} "
                  f"- {country} will be missing from data/raw/. Check the extracted tree by hand "
                  f"(e.g. `find {extract_root} -iname '*{country.split('_')[0]}*' -type d`) and symlink "
                  f"it manually if this script guessed wrong.")
            continue

        min_depth = min(len(p.relative_to(extract_root).parts) for p in candidates)
        shallowest = sorted(p for p in candidates if len(p.relative_to(extract_root).parts) == min_depth)
        if len(shallowest) > 1:
            print(f"  NOTE: multiple equally-shallow matches for {country}, picking the first:")
            for p in shallowest:
                print(f"    - {p.relative_to(extract_root)}")
        chosen = shallowest[0]
        link_path.symlink_to(chosen, target_is_directory=True)
        print(f"  {country}: linked data/raw/{country} -> {chosen.relative_to(extract_root)}")
        found_any = True

    if not found_any:
        print("  WARNING: none of the requested countries were found - inspect the extracted tree "
              f"under {extract_root} directly; the assumed alias names in COUNTRY_ALIASES may not "
              "match this zip's actual layout.")


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR),
                         help="root data directory (default: %(default)s)")
    parser.add_argument("--countries", nargs="+", choices=list(COUNTRY_ALIASES) + ["China"], default=None,
                         help="subset of countries to link into data/raw/ after extraction (default: all). "
                              "Does NOT reduce download size - FigShare ships one combined zip for all "
                              "six countries. Pass 'China' for both China_MotorBike and China_Drone.")
    parser.add_argument("--skip-extract", action="store_true", help="download (+ verify) only, don't unzip")
    parser.add_argument("--connections", type=int, default=16,
                         help="parallel connections for aria2c downloads (default: %(default)s, ignored "
                              "if aria2c isn't installed)")
    args = parser.parse_args(argv)

    countries = args.countries
    if countries is None:
        countries = list(COUNTRY_ALIASES)
    elif "China" in countries:
        countries = [c for c in countries if c != "China"] + ["China_MotorBike", "China_Drone"]

    data_dir = Path(args.data_dir)
    zips_dir = data_dir / "zips"
    extract_root = data_dir / "raw" / "_extracted_all"
    raw_dir = data_dir / "raw"

    zip_path = zips_dir / FIGSHARE_ZIP_NAME
    print(f"Downloading combined RDD2022 archive (~12.35GB) from FigShare into {zip_path}")
    download_one(FIGSHARE_URL, zip_path, connections=args.connections)

    if not verify_md5(zip_path, FIGSHARE_MD5):
        print("\nAborting - downloaded file failed MD5 verification. Delete the zip and re-run.")
        return 1

    if args.skip_extract:
        print("\n--skip-extract set, stopping after download+verify.")
        return 0

    print(f"\nExtracting into {extract_root}")
    extract_one(zip_path, extract_root)

    print(f"\nLinking {len(countries)} countries into {raw_dir}")
    reorganize_countries(extract_root, raw_dir, countries)

    print("\nDone. Raw data is under:", raw_dir)
    print("Next: python scripts/convert_voc_to_yolo.py")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/convert_voc_to_yolo.py
"""Convert RDD2022's PASCAL-VOC-XML annotations into YOLO format, and build a unified manifest
across all six countries (roadmap: infra-defect-detection, phase 1).

Why this exists rather than just pointing Ultralytics at the raw VOC XML: (1) YOLO training wants
one .txt label per image with normalized [class cx cy w h] rows, not VOC's per-object XML; (2) more
importantly for this project, we need a single manifest that records which COUNTRY every image
came from, because phase 3's whole point is training on a subset of countries and evaluating
cross-country generalization - that experiment is impossible without country labels surviving the
conversion step.

Design choice - discover files by globbing + matching by filename stem, not by assuming a fixed
"images/" + "annotations/xmls/" subfolder layout: RDD2022's own directory-structure reference file
was unreachable when this project was set up (see download_rdd2022.py's docstring), so hardcoding
an assumed layout would risk silently processing zero files if the real layout differs. Globbing
recursively is slower but correct regardless of how each country's zip is actually organized
internally.

Damage classes (from the RDD2022/CRDDC'2022 label map):
    D00 - longitudinal crack
    D10 - transverse crack
    D20 - alligator crack
    D40 - pothole
Any other class name encountered (older RDD releases had more, e.g. D01/D11/D43/D44/D50) is logged
and SKIPPED, not silently merged into the nearest class - if you see a nontrivial skip count for a
country, that's worth a manual look before assuming the data converted cleanly.

Usage:
    python scripts/convert_voc_to_yolo.py                  # converts every country under data/raw/
    python scripts/convert_voc_to_yolo.py --countries Japan Czech
"""
import argparse
import csv
import sys
import xml.etree.ElementTree as ET
from pathlib import Path

try:
    from PIL import Image
except ImportError:
    Image = None

DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"

CLASS_MAP = {"D00": 0, "D10": 1, "D20": 2, "D40": 3}
CLASS_NAMES = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole"]


def find_files(root, suffix):
    return sorted(p for p in root.rglob(f"*{suffix}") if p.is_file())


def parse_voc_xml(xml_path):
    """Return (image_filename, width, height, [(class_name, xmin, ymin, xmax, ymax), ...]).

    width/height come from the XML's <size> block when present; if absent or zero (seen in some
    messy real-world VOC exports), the caller falls back to opening the image with PIL - this is
    exactly the kind of "annotation doesn't quite match what a clean benchmark would give you"
    messiness the project is supposed to be diagnosing, so we handle it rather than crash on it.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename_el = root.find("filename")
    image_filename = filename_el.text.strip() if filename_el is not None and filename_el.text else xml_path.stem + ".jpg"

    size_el = root.find("size")
    width = height = 0
    if size_el is not None:
        w_el, h_el = size_el.find("width"), size_el.find("height")
        width = int(w_el.text) if w_el is not None and w_el.text else 0
        height = int(h_el.text) if h_el is not None and h_el.text else 0

    objects = []
    for obj in root.findall("object"):
        name_el = obj.find("name")
        bnd = obj.find("bndbox")
        if name_el is None or bnd is None:
            continue
        name = (name_el.text or "").strip()
        try:
            xmin = float(bnd.find("xmin").text)
            ymin = float(bnd.find("ymin").text)
            xmax = float(bnd.find("xmax").text)
            ymax = float(bnd.find("ymax").text)
        except (AttributeError, TypeError, ValueError):
            continue
        objects.append((name, xmin, ymin, xmax, ymax))

    return image_filename, width, height, objects


def voc_box_to_yolo_line(class_idx, xmin, ymin, xmax, ymax, width, height):
    cx = (xmin + xmax) / 2 / width
    cy = (ymin + ymax) / 2 / height
    w = (xmax - xmin) / width
    h = (ymax - ymin) / height
    return f"{class_idx} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def convert_country(country, raw_dir, processed_dir, manifest_rows, stats):
    xml_files = find_files(raw_dir, ".xml")
    jpg_files = find_files(raw_dir, ".jpg") + find_files(raw_dir, ".jpeg") + find_files(raw_dir, ".JPG")
    image_by_stem = {}
    for p in jpg_files:
        image_by_stem.setdefault(p.stem, p)

    if not xml_files:
        print(f"  WARNING: no .xml annotation files found under {raw_dir} - is the zip actually extracted here?")
        return

    out_images = processed_dir / country / "images"
    out_labels = processed_dir / country / "labels"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    n_ok = n_orphan_xml = n_bad_size = n_no_objects = n_skipped_class = 0

    for xml_path in xml_files:
        image_filename, width, height, objects = parse_voc_xml(xml_path)
        stem = Path(image_filename).stem

        image_path = image_by_stem.get(stem) or image_by_stem.get(xml_path.stem)
        if image_path is None:
            n_orphan_xml += 1
            continue

        if width <= 0 or height <= 0:
            if Image is None:
                n_bad_size += 1
                continue
            try:
                with Image.open(image_path) as im:
                    width, height = im.size
            except Exception:
                n_bad_size += 1
                continue

        yolo_lines = []
        for name, xmin, ymin, xmax, ymax in objects:
            if name not in CLASS_MAP:
                n_skipped_class += 1
                stats["skipped_classes"][name] = stats["skipped_classes"].get(name, 0) + 1
                continue
            xmin, xmax = sorted((max(0, xmin), min(width, xmax)))
            ymin, ymax = sorted((max(0, ymin), min(height, ymax)))
            if xmax <= xmin or ymax <= ymin:
                continue
            yolo_lines.append(voc_box_to_yolo_line(CLASS_MAP[name], xmin, ymin, xmax, ymax, width, height))
            stats["class_counts"][name] = stats["class_counts"].get(name, 0) + 1

        if not yolo_lines:
            n_no_objects += 1
            continue

        dest_stem = f"{country}__{stem}"
        dest_image = out_images / f"{dest_stem}{image_path.suffix.lower()}"
        dest_label = out_labels / f"{dest_stem}.txt"
        if not dest_image.exists():
            dest_image.write_bytes(image_path.read_bytes())
        dest_label.write_text("\n".join(yolo_lines) + "\n")

        manifest_rows.append({
            "country": country,
            "image": str(dest_image.relative_to(processed_dir.parent)),
            "label": str(dest_label.relative_to(processed_dir.parent)),
            "width": width,
            "height": height,
            "num_objects": len(yolo_lines),
            "classes": ";".join(sorted({l.split()[0] for l in yolo_lines})),
        })
        n_ok += 1

    print(f"  {country}: {n_ok} converted, {n_orphan_xml} orphan xml (no matching image), "
          f"{n_bad_size} bad/unreadable size, {n_no_objects} had zero valid objects after class "
          f"filtering, {n_skipped_class} individual boxes skipped for unrecognized class names")
    stats["per_country"][country] = {
        "converted": n_ok, "orphan_xml": n_orphan_xml, "bad_size": n_bad_size,
        "no_objects": n_no_objects, "skipped_class_boxes": n_skipped_class,
    }


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR))
    parser.add_argument("--countries", nargs="+", default=None,
                         help="subset of country folder names under data/raw/ to convert (default: all found)")
    args = parser.parse_args(argv)

    if Image is None:
        print("NOTE: Pillow not installed - images with missing/zero <size> in their XML will be "
              "skipped instead of measured. `pip install Pillow` to handle those too.", file=sys.stderr)

    data_dir = Path(args.data_dir)
    raw_dir = data_dir / "raw"
    processed_dir = data_dir / "processed"

    if args.countries:
        countries = args.countries
    else:
        countries = sorted(p.name for p in raw_dir.iterdir() if p.is_dir())

    if not countries:
        print(f"No country folders found under {raw_dir} - run download_rdd2022.py first.", file=sys.stderr)
        return 1

    manifest_rows = []
    stats = {"class_counts": {}, "skipped_classes": {}, "per_country": {}}

    print(f"Converting {len(countries)} countries: {countries}")
    for country in countries:
        print(f"\n[{country}]")
        convert_country(country, raw_dir / country, processed_dir, manifest_rows, stats)

    manifest_path = data_dir / "manifest.csv"
    with open(manifest_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["country", "image", "label", "width", "height", "num_objects", "classes"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    dataset_yaml = data_dir / "dataset.yaml"
    dataset_yaml.write_text(
        "# Auto-generated by convert_voc_to_yolo.py - combined view across all converted countries.\n"
        "# For the phase-3 cross-country experiments, build per-experiment yaml files that point at\n"
        "# a subset of countries' image folders instead of reusing this combined one directly.\n"
        f"path: {processed_dir}\n"
        "train: */images\n"
        f"nc: {len(CLASS_NAMES)}\n"
        f"names: {CLASS_NAMES}\n"
    )

    print(f"\nWrote manifest ({len(manifest_rows)} images) to {manifest_path}")
    print(f"Wrote combined dataset.yaml to {dataset_yaml}")
    print("\nClass distribution across all converted countries:")
    for name, idx in CLASS_MAP.items():
        print(f"  {name}: {stats['class_counts'].get(name, 0)}")
    if stats["skipped_classes"]:
        print("\nSkipped (unrecognized) class names encountered - investigate before trusting counts above:")
        for name, count in sorted(stats["skipped_classes"].items(), key=lambda kv: -kv[1]):
            print(f"  {name!r}: {count}")

    print("\nNext: python scripts/eda_report.py")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/eda_report.py
"""Quantify how the six RDD2022 countries differ (roadmap: infra-defect-detection, phase 1).

This is the deliverable phase 1 is actually for: not "look at some pictures" but a report that puts
numbers on exactly which properties differ across countries, so phase 3's cross-country
generalization experiment has a documented hypothesis to test rather than a vague "the data is
probably different" assumption. Reads data/manifest.csv (written by convert_voc_to_yolo.py) and
the images/labels it points at.

Computes, per country:
  - image count, object count, objects-per-image
  - class distribution (proportion of D00/D10/D20/D40)
  - resolution distribution (most common width x height, and how much it varies)
  - brightness (mean grayscale pixel value, sampled - see --sample-size) as a cheap proxy for
    lighting-condition differences (weather, time of day, camera exposure settings)

Deliberately NOT trying to detect "blur" or "occlusion" quantitatively here - those need either a
trained model's failure cases (that's phase 3, once there's a baseline to probe) or a much more
careful CV pipeline (Laplacian-variance blur detection is noisy on road-texture images specifically,
since sharp asphalt texture can score similarly to blur). Brightness and resolution are the two
properties measurable directly from pixels without a model in the loop, so that's what phase 1
reports; phase 3's failure-case analysis is where blur/occlusion get characterized properly, on the
* subset that actually caused cross-country failures* rather than on the whole dataset speculatively.

Usage:
    python scripts/eda_report.py                    # samples 200 images/country for brightness
    python scripts/eda_report.py --sample-size 500   # slower, more precise brightness estimate
"""
import argparse
import random
from collections import Counter
from pathlib import Path

import pandas as pd

try:
    import numpy as np
    from PIL import Image
    HAVE_IMAGING = True
except ImportError:
    HAVE_IMAGING = False

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except ImportError:
    HAVE_MPL = False

DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"
CLASS_NAMES = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole"]


def sample_brightness(image_paths, sample_size, seed=42):
    """Mean/std of grayscale brightness over a random sample of image_paths, resized small first
    purely for speed - brightness is a coarse global statistic, a 64x64 downsample is plenty."""
    if not HAVE_IMAGING:
        return None, None, 0
    rng = random.Random(seed)
    sample = rng.sample(image_paths, min(sample_size, len(image_paths)))
    values = []
    for p in sample:
        try:
            with Image.open(p) as im:
                im = im.convert("L").resize((64, 64))
                values.append(np.asarray(im, dtype="float32").mean())
        except Exception:
            continue
    if not values:
        return None, None, 0
    arr = np.array(values)
    return float(arr.mean()), float(arr.std()), len(values)


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR))
    parser.add_argument("--sample-size", type=int, default=200,
                         help="images per country to sample for the brightness statistic (default: %(default)s)")
    args = parser.parse_args(argv)

    data_dir = Path(args.data_dir)
    manifest_path = data_dir / "manifest.csv"
    if not manifest_path.exists():
        print(f"{manifest_path} not found - run convert_voc_to_yolo.py first.")
        return 1

    df = pd.read_csv(manifest_path)
    df["aspect_ratio"] = df["width"] / df["height"]

    rows = []
    figures_dir = data_dir / "eda_figures"
    figures_dir.mkdir(exist_ok=True)

    class_dist_by_country = {}
    for country, group in df.groupby("country"):
        class_counter = Counter()
        for classes_field in group["classes"].fillna(""):
            for c in str(classes_field).split(";"):
                if c:
                    class_counter[int(c)] += 1
        total_objects = sum(class_counter.values())
        class_dist_by_country[country] = {
            CLASS_NAMES[i]: class_counter.get(i, 0) / total_objects if total_objects else 0.0
            for i in range(len(CLASS_NAMES))
        }

        res_counts = group.groupby(["width", "height"]).size().sort_values(ascending=False)
        top_res = res_counts.index[0] if len(res_counts) else (None, None)
        res_diversity = len(res_counts)

        image_paths = [data_dir / p for p in group["image"]]
        brightness_mean, brightness_std, n_sampled = sample_brightness(image_paths, args.sample_size)

        rows.append({
            "country": country,
            "n_images": len(group),
            "n_objects": int(group["num_objects"].sum()),
            "objects_per_image": group["num_objects"].mean(),
            "top_resolution": f"{top_res[0]}x{top_res[1]}" if top_res[0] else "unknown",
            "distinct_resolutions": res_diversity,
            "mean_aspect_ratio": group["aspect_ratio"].mean(),
            "brightness_mean": brightness_mean,
            "brightness_std": brightness_std,
            "brightness_n_sampled": n_sampled,
        })

    summary = pd.DataFrame(rows).sort_values("country")
    summary_path = data_dir / "eda_summary.csv"
    summary.to_csv(summary_path, index=False)

    if HAVE_MPL:
        fig, ax = plt.subplots(figsize=(10, 5))
        countries = summary["country"].tolist()
        x = range(len(countries))
        width_bar = 0.2
        for i, name in enumerate(CLASS_NAMES):
            values = [class_dist_by_country[c][name] for c in countries]
            ax.bar([xi + i * width_bar for xi in x], values, width=width_bar, label=name)
        ax.set_xticks([xi + 1.5 * width_bar for xi in x])
        ax.set_xticklabels(countries, rotation=30, ha="right")
        ax.set_ylabel("share of annotated objects")
        ax.set_title("Damage class distribution by country")
        ax.legend()
        fig.tight_layout()
        fig.savefig(figures_dir / "class_distribution_by_country.png", dpi=150)
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(summary["country"], summary["brightness_mean"], yerr=summary["brightness_std"])
        ax.set_ylabel("mean grayscale brightness (0-255)")
        ax.set_title(f"Brightness by country (sampled, n<= {args.sample_size}/country)")
        plt.xticks(rotation=30, ha="right")
        fig.tight_layout()
        fig.savefig(figures_dir / "brightness_by_country.png", dpi=150)
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(summary["country"], summary["n_images"])
        ax.set_ylabel("images")
        ax.set_title("Image count by country")
        plt.xticks(rotation=30, ha="right")
        fig.tight_layout()
        fig.savefig(figures_dir / "image_count_by_country.png", dpi=150)
        plt.close(fig)
    else:
        print("matplotlib not installed - skipping figure generation, CSV/markdown still written.")

    report_lines = [
        "# RDD2022 six-country data profile\n",
        "Generated by `scripts/eda_report.py`. This is the phase-1 deliverable: quantifying how the "
        "six countries' data actually differs, as the documented basis for the phase-3 cross-country "
        "generalization experiment - not an assumption.\n",
        "## Summary table\n",
        summary.to_markdown(index=False) if hasattr(summary, "to_markdown") else summary.to_string(index=False),
        "\n\n## Class distribution by country (share of annotated objects)\n",
    ]
    class_dist_df = pd.DataFrame(class_dist_by_country).T[CLASS_NAMES]
    report_lines.append(class_dist_df.to_markdown() if hasattr(class_dist_df, "to_markdown") else class_dist_df.to_string())
    report_lines.append(
        "\n\n## Figures\n\n"
        "- `eda_figures/class_distribution_by_country.png`\n"
        "- `eda_figures/brightness_by_country.png`\n"
        "- `eda_figures/image_count_by_country.png`\n"
    )
    report_path = data_dir / "eda_report.md"
    report_path.write_text("\n".join(report_lines))

    print(summary.to_string(index=False))
    print(f"\nWrote {summary_path}")
    print(f"Wrote {report_path}")
    if HAVE_MPL:
        print(f"Wrote figures to {figures_dir}")
    print("\nNext: read eda_report.md, note which countries look most different from the rest on "
          "brightness/resolution/class-mix, then design the phase-3 train/test country split around "
          "those specific differences instead of an arbitrary split.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/extract_convert_per_country.py
"""Extract + convert RDD2022 ONE COUNTRY AT A TIME, deleting each country's intermediate data as
soon as it's converted (roadmap: infra-defect-detection, phase 1 - Kaggle disk-constrained variant).

WHY THIS EXISTS (2026-09-17, v2->v3): the straightforward approach - download_rdd2022.py's normal
extract_one()/reorganize_countries(), which calls zf.extractall() on the WHOLE combined zip at once
- needs the 12.35GB combined zip AND all the extracted raw VOC data on disk simultaneously, which
blew through a Kaggle notebook's working-directory quota on the first attempt.

v2 tried to fix this by extracting one country's files at a time straight out of the combined zip's
member list - but that assumed the combined zip was a flat tree of images/XML per country. It
ISN'T: FigShare's combined zip (b62bd51d2ffcfaa76c60f234f0cc2bb3, the officially-published MD5) is
actually a ZIP-OF-ZIPS - exactly 7 entries, one per country, each itself a complete nested .zip
(confirmed 2026-09-17 by actually listing the real zip's namelist on Kaggle: `RDD2022/Japan.zip`,
`RDD2022/India.zip`, `RDD2022/Czech.zip`, `RDD2022/Norway.zip`, `RDD2022/United_States.zip`,
`RDD2022/China_MotorBike.zip`, `RDD2022/China_Drone.zip`). v2's matching logic looked for a path
COMPONENT exactly equal to a country alias (e.g. a directory literally named "Japan"), which never
matched because the actual component is "Japan.zip" (a file, not a directory) - so v2 silently
converted 0 images across all 7 countries and wrote an empty manifest.csv.

v3 (this version) matches each entry by its FILENAME STEM (`Path("RDD2022/Japan.zip").stem ==
"Japan"`) instead, then per country: copies just that country's nested zip out of the combined zip
to a small temp file, opens THAT as its own zip and extracts its actual images/XML from it, converts
those, then deletes the temp nested zip AND the extracted raw folder before moving to the next
country. Peak disk per country = combined zip (12.35GB, constant throughout) + one country's nested
zip (a few GB at most) + that same country's extracted raw + that country's converted/processed
data - never all seven countries' data at once.

Usage:
    python scripts/extract_convert_per_country.py                  # all countries, then deletes the combined zip
    python scripts/extract_convert_per_country.py --keep-zip       # don't delete the combined zip at the end
    python scripts/extract_convert_per_country.py --data-dir data
"""
import argparse
import csv
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import download_rdd2022 as dl          # noqa: E402  (FIGSHARE_ZIP_NAME, COUNTRY_ALIASES)
import convert_voc_to_yolo as cv       # noqa: E402  (convert_country, CLASS_MAP, CLASS_NAMES)


def _format_bytes(n):
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}TB"


DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"


def _print_disk_usage(label):
    result = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
    print(f"  [{label}] disk usage:\n" + "\n".join("    " + l for l in result.stdout.splitlines()))


def match_country_by_stem(entry_name):
    """Which COUNTRY_ALIASES key this combined-zip entry is, by comparing its filename stem (the
    name with the LAST extension stripped, e.g. "RDD2022/China_MotorBike.zip" -> "China_MotorBike")
    against the known aliases, case-insensitively. This is the v3 fix: v2 matched whole path
    COMPONENTS looking for a directory named e.g. "Japan", which never matched because the real
    entries are files named "Japan.zip", not directories named "Japan"."""
    stem_lower = Path(entry_name).stem.lower()
    for country, aliases in dl.COUNTRY_ALIASES.items():
        for alias in aliases:
            if alias.lower() == stem_lower:
                return country
    return None


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR))
    parser.add_argument("--keep-zip", action="store_true", help="don't delete the combined zip once done (default: delete it)")
    args = parser.parse_args(argv)

    data_dir = Path(args.data_dir)
    zip_path = data_dir / "zips" / dl.FIGSHARE_ZIP_NAME
    raw_dir = data_dir / "raw"
    processed_dir = data_dir / "processed"

    if not zip_path.exists():
        print(f"{zip_path} not found - run download_rdd2022.py --skip-extract first.", file=sys.stderr)
        return 1

    raw_dir.mkdir(parents=True, exist_ok=True)
    processed_dir.mkdir(parents=True, exist_ok=True)

    manifest_rows = []
    stats = {"class_counts": {}, "skipped_classes": {}, "per_country": {}}

    print(f"Opening {zip_path} to read its member list (no extraction yet, costs no disk space)...")
    with zipfile.ZipFile(zip_path) as outer_zf:
        infos = [i for i in outer_zf.infolist() if not i.filename.endswith("/")]
        print(f"  combined zip contains {len(infos)} entries")

        by_country = {}
        unmatched = []
        for info in infos:
            country = match_country_by_stem(info.filename)
            if country:
                by_country[country] = info
            else:
                unmatched.append(info.filename)

        print("\nEntries matched (this is the combined zip's REAL layout - one nested zip per country):")
        for country in dl.COUNTRY_ALIASES:
            if country in by_country:
                info = by_country[country]
                print(f"  {country}: {info.filename} ({_format_bytes(info.file_size)})")
            else:
                print(f"  {country}: NOT FOUND")
        if unmatched:
            print(f"\n  {len(unmatched)} entries matched no known country alias:")
            for n in unmatched:
                print(f"    {n}")

        _print_disk_usage("before extracting any country")

        for country, info in by_country.items():
            country_raw = raw_dir / country
            if country_raw.exists():
                shutil.rmtree(country_raw)
            country_raw.mkdir(parents=True)

            nested_zip_tmp = raw_dir / f"_nested_{country}.zip"
            print(f"\n[{country}] copying nested zip {info.filename} ({_format_bytes(info.file_size)}) out to {nested_zip_tmp}...")
            with outer_zf.open(info) as src, open(nested_zip_tmp, "wb") as dst:
                shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)

            print(f"  extracting nested zip into {country_raw}...")
            with zipfile.ZipFile(nested_zip_tmp) as inner_zf:
                bad = inner_zf.testzip()
                if bad is not None:
                    print(f"  WARNING: {nested_zip_tmp} is corrupt (bad member: {bad}) - skipping {country}")
                    nested_zip_tmp.unlink()
                    shutil.rmtree(country_raw)
                    continue
                inner_zf.extractall(country_raw)

            print(f"  deleting nested zip (already extracted, no longer needed)...")
            nested_zip_tmp.unlink()

            print(f"  converting...")
            cv.convert_country(country, country_raw, processed_dir, manifest_rows, stats)

            print(f"  deleting {country_raw} (already converted, no longer needed) to free space for the next country...")
            shutil.rmtree(country_raw)
            _print_disk_usage(f"after {country}")

    manifest_path = data_dir / "manifest.csv"
    with open(manifest_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["country", "image", "label", "width", "height", "num_objects", "classes"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    dataset_yaml = data_dir / "dataset.yaml"
    dataset_yaml.write_text(
        "# Auto-generated by extract_convert_per_country.py - combined view across all converted countries.\n"
        "# For the phase-3 cross-country experiments, build per-experiment yaml files that point at\n"
        "# a subset of countries' image folders instead of reusing this combined one directly.\n"
        f"path: {processed_dir}\n"
        "train: */images\n"
        f"nc: {len(cv.CLASS_NAMES)}\n"
        f"names: {cv.CLASS_NAMES}\n"
    )

    print(f"\nWrote manifest ({len(manifest_rows)} images) to {manifest_path}")
    print(f"Wrote combined dataset.yaml to {dataset_yaml}")
    print("\nClass distribution across all converted countries:")
    for name in cv.CLASS_MAP:
        print(f"  {name}: {stats['class_counts'].get(name, 0)}")
    if stats["skipped_classes"]:
        print("\nSkipped (unrecognized) class names encountered - investigate before trusting counts above:")
        for name, count in sorted(stats["skipped_classes"].items(), key=lambda kv: -kv[1]):
            print(f"  {name!r}: {count}")

    if not manifest_rows:
        print("\nWARNING: manifest is EMPTY - 0 images were converted. Check the 'Entries matched' listing "
              "above for NOT FOUND countries or unmatched entries before trusting anything downstream.")

    if not args.keep_zip:
        zip_path.unlink()
        print(f"\nDeleted {zip_path} - every country has been extracted and converted from it already.")
    else:
        print(f"\n--keep-zip set: leaving {zip_path} in place.")

    _print_disk_usage("final")
    print("\nNext: python scripts/eda_report.py")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 2. 安装依赖

In [ ]:
!pip install -q "pandas>=2.0" "Pillow>=10.0" "matplotlib>=3.7" "numpy>=1.24" "tabulate>=0.9"


## 3. 下载 + MD5校验(只下载,不解压)

**如果`data/zips/`里已经有校验过的zip了(第0步能看到),这一步会直接跳过下载**,不用担心重跑。

In [ ]:
!python scripts/download_rdd2022.py --skip-extract


## 4. 按国家分批:复制出嵌套zip→解压→转换→删除中间文件→下一国

这一步会先打印匹配到的7个国家entry(文件名+大小),确认都不是`NOT FOUND`再继续。每处理完一个国家会打印一次磁盘用量。全部处理完、manifest.csv和dataset.yaml写好之后才会删除外层的12.35GB zip。

**如果这一步打印出的manifest不是0张图,就说明这次是真的转换成功了**,可以放心跑第5步的EDA。

In [ ]:
!python scripts/extract_convert_per_country.py


## 5. 跨国EDA,生成 `data/eda_summary.csv` + `data/eda_report.md` + `data/eda_figures/`

In [ ]:
!python scripts/eda_report.py


## 6. 核对结果(对应交接文档里"实际去核实文件内容"这条标准步骤)

In [ ]:
!echo '--- data/raw (应为空或不存在) ---'
!ls -la data/raw/ 2>&1
!echo '--- du -sh data/* ---'
!du -sh data/* 2>/dev/null
!echo '--- manifest rows (应该远大于1,7国合计) ---'
!wc -l data/manifest.csv
!echo '--- 按国家数一下manifest里的行数,7国都应该有 ---'
!cut -d, -f1 data/manifest.csv | sort | uniq -c
!echo '--- eda_report.md ---'
!cat data/eda_report.md
!df -h /


## 7. 收尾:提交并把小文件带回Mac

1. 点右上角 **Save Version → Save & Run All (Commit)**。
2. commit跑完后,打开这个版本的 **Output** 标签页,只下载这几个小文件到本地:
   - `data/manifest.csv`
   - `data/eda_summary.csv`
   - `data/eda_report.md`
   - `data/eda_figures/*.png`(3张图)
3. 把下载下来的这几个文件放进Mac上 `~/PycharmProjects/infra-defect-detection/data/` 对应位置,告诉我一声,我再帮你核对一下文件内容和数量对不对(尤其是7个国家是不是都有数据,不是只有部分国家)。
4. `data/processed/`留在这个notebook的Output里,Phase 2的YOLO11训练notebook可以直接把它加成一个数据源用。
